# 02 — Wait Time Prediction Model
Train and compare 3 supervised regression models to predict OPD wait time (`WaitTimeMinutes`):
1. **Linear Regression** — simple linear baseline
2. **K-Nearest Neighbors (KNN)** — instance-based, distance-driven model, tuned via GridSearchCV
3. **Random Forest Regressor** — tree-ensemble model, tuned via GridSearchCV

These three represent genuinely different algorithm families — linear, distance-based, and tree-ensemble — so the comparison reflects real differences in how each approach handles this data, not just noise.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

df = pd.read_csv('../data/processed/opd_visits_clean.csv')
df.head()

In [ ]:
# Features and target
categorical_features = ['Department', 'DayOfWeek', 'TriageCategory', 'AppointmentType']
numeric_features = ['ArrivalHour', 'IsWeekend', 'FacilityOccupancyRate', 'ProvidersOnShift', 'NursesOnShift', 'StaffToPatientRatio']
target = 'WaitTimeMinutes'

features = categorical_features + numeric_features
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

In [ ]:
# Shared preprocessing: one-hot encode categorical columns, pass numeric columns through as-is
preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)],
    remainder='passthrough'
)

## Model 1 — Linear Regression (baseline, no tuning needed)

In [ ]:
lr_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2 = r2_score(y_test, lr_preds)

print(f'Linear Regression -> MAE: {lr_mae:.2f} | RMSE: {lr_rmse:.2f} | R2: {lr_r2:.3f}')

## Model 2 — K-Nearest Neighbors (GridSearchCV tuned)
KNN predicts a patient's wait time by averaging the wait times of the most similar past patients (their "neighbors"). We search over the number of neighbors (k) and the weighting scheme to find its best fair configuration.

In [ ]:
knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', KNeighborsRegressor())
])

knn_param_grid = {
    'regressor__n_neighbors': [5, 10, 15, 20, 30, 50],
    'regressor__weights': ['uniform', 'distance'],
}

knn_search = GridSearchCV(
    knn_pipeline, knn_param_grid, cv=3,
    scoring='neg_mean_absolute_error', n_jobs=-1
)
knn_search.fit(X_train, y_train)

print('Best KNN params:', knn_search.best_params_)
knn_model = knn_search.best_estimator_
knn_preds = knn_model.predict(X_test)

knn_mae = mean_absolute_error(y_test, knn_preds)
knn_rmse = np.sqrt(mean_squared_error(y_test, knn_preds))
knn_r2 = r2_score(y_test, knn_preds)

print(f'KNN (tuned) -> MAE: {knn_mae:.2f} | RMSE: {knn_rmse:.2f} | R2: {knn_r2:.3f}')

## Model 3 — Random Forest Regressor (GridSearchCV tuned)

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

rf_param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [4, 5, 6, 8, 12],
    'regressor__min_samples_leaf': [1, 5, 10],
}

rf_search = GridSearchCV(
    rf_pipeline, rf_param_grid, cv=3,
    scoring='neg_mean_absolute_error', n_jobs=-1
)
rf_search.fit(X_train, y_train)

print('Best Random Forest params:', rf_search.best_params_)
rf_model = rf_search.best_estimator_
rf_preds = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)

print(f'Random Forest (tuned) -> MAE: {rf_mae:.2f} | RMSE: {rf_rmse:.2f} | R2: {rf_r2:.3f}')

## Model Comparison (with Accuracy metric)
In addition to MAE/RMSE/R2, we add **Accuracy (% within ±15 min)** — the share of predictions that land within 15 minutes of the true wait time. This is a more intuitive, stakeholder-friendly metric than R2 alone.

In [ ]:
def accuracy_within(y_true, y_pred, threshold=15):
    return np.mean(np.abs(y_true.values - y_pred) <= threshold) * 100

results = pd.DataFrame({
    'Model': ['Linear Regression', 'KNN', 'Random Forest'],
    'MAE': [lr_mae, knn_mae, rf_mae],
    'RMSE': [lr_rmse, knn_rmse, rf_rmse],
    'R2': [lr_r2, knn_r2, rf_r2],
    'Accuracy_pct': [
        accuracy_within(y_test, lr_preds),
        accuracy_within(y_test, knn_preds),
        accuracy_within(y_test, rf_preds),
    ],
}).round(3)

results = results.sort_values('MAE')
results

In [ ]:
# Visualize the comparison
fig, axes = plt.subplots(1, 3, figsize=(16,4))

axes[0].bar(results['Model'], results['MAE'], color=['#B8A1E3','#8B5FBF','#4B2E83'])
axes[0].set_title('MAE by Model (lower is better)')
axes[0].set_ylabel('Minutes')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(results['Model'], results['R2'], color=['#B8A1E3','#8B5FBF','#4B2E83'])
axes[1].set_title('R2 Score by Model (higher is better)')
axes[1].tick_params(axis='x', rotation=15)

axes[2].bar(results['Model'], results['Accuracy_pct'], color=['#B8A1E3','#8B5FBF','#4B2E83'])
axes[2].set_title('Accuracy % (within ±15 min)')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../reports/screenshots/model_comparison.png', dpi=150)
plt.show()

## Which Model Won, and Why?

In [ ]:
best_by_mae = results.iloc[0]
worst_by_mae = results.iloc[-1]
improvement_pct = (1 - best_by_mae['MAE'] / worst_by_mae['MAE']) * 100

print('=' * 65)
print('MODEL COMPARISON VERDICT')
print('=' * 65)
for _, row in results.iterrows():
    print(f"{row['Model']:<20} MAE: {row['MAE']:>6.2f} min | R2: {row['R2']:.3f} | "
          f"Accuracy: {row['Accuracy_pct']:.1f}%")
print('-' * 65)
print(f"WINNER: {best_by_mae['Model']}")
print(f"  -> Predictions are off by {best_by_mae['MAE']:.1f} minutes on average.")
print(f"  -> {improvement_pct:.1f}% more accurate (lower MAE) than the weakest model ({worst_by_mae['Model']}).")
print(f"  -> {best_by_mae['Accuracy_pct']:.1f}% of predictions fall within ±15 minutes of the true wait time.")
print()
print(f"WHY KNN STRUGGLED: KNN measures 'distance' between patients across many")
print(f"one-hot encoded categorical dimensions (Department, Triage, etc). With so")
print(f"many dimensions, the notion of a 'close neighbor' becomes less meaningful —")
print(f"a well-known phenomenon called the 'Curse of Dimensionality'.")
print('=' * 65)

## Actual vs. Predicted — Line Chart Comparison

In [ ]:
sample_size = 60
sample_idx = y_test.reset_index(drop=True).sample(sample_size, random_state=42).sort_values().index

actual_sorted = y_test.reset_index(drop=True).iloc[sample_idx].values
lr_sorted = pd.Series(lr_preds).iloc[sample_idx].values
knn_sorted = pd.Series(knn_preds).iloc[sample_idx].values
rf_sorted = pd.Series(rf_preds).iloc[sample_idx].values

plt.figure(figsize=(12, 6))
plt.plot(actual_sorted, label='Actual Wait Time', color='#2D1B4E', linewidth=2.5)
plt.plot(lr_sorted, label=f'Linear Regression (MAE={lr_mae:.1f})', color='#B8A1E3', linestyle='--', marker='o', markersize=3)
plt.plot(knn_sorted, label=f'KNN (MAE={knn_mae:.1f})', color='#8B5FBF', linestyle='--', marker='s', markersize=3)
plt.plot(rf_sorted, label=f'Random Forest (MAE={rf_mae:.1f})', color='#4B2E83', linestyle='--', marker='^', markersize=3)

plt.title('Actual vs. Predicted Wait Time — Model Comparison (sample of test patients)')
plt.xlabel('Patient (sorted by actual wait time, low to high)')
plt.ylabel('Wait Time (minutes)')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/screenshots/actual_vs_predicted_line.png', dpi=150)
plt.show()

## Feature Importance (from Random Forest)

In [ ]:
feature_names = rf_model.named_steps['preprocessor'].get_feature_names_out()
importances = rf_model.named_steps['regressor'].feature_importances_

top_n = 15
idx = np.argsort(importances)[-top_n:]
plt.figure(figsize=(8,6))
plt.barh(range(top_n), importances[idx], color='#8B5FBF')
plt.yticks(range(top_n), [feature_names[i] for i in idx])
plt.title('Top Feature Importances (Random Forest)')
plt.tight_layout()
plt.savefig('../reports/screenshots/feature_importance.png', dpi=150)
plt.show()

## Save the Best Model

In [ ]:
all_models = {'Linear Regression': (lr_model, lr_mae), 'KNN': (knn_model, knn_mae), 'Random Forest': (rf_model, rf_mae)}
best_name = min(all_models, key=lambda k: all_models[k][1])
best_model = all_models[best_name][0]

joblib.dump(best_model, '../models/wait_time_model.pkl')
print(f'Best model: {best_name} (MAE = {all_models[best_name][1]:.2f} minutes)')
print('Saved to models/wait_time_model.pkl')